In [ ]:
# start coding here
from cellwhisperer.config import get_path

In [ ]:
# start coding here

import matplotlib.pyplot as plt
import seaborn
import matplotlib
import pandas as pd

df = pd.read_csv(snakemake.input.csv, index_col=0)
df = df.loc[:, snakemake.params.metrics]
df.rename(columns={"valfn_daniel_strictly_deduplicated_dmis-lab_biobert-v1.1_CLS_pooling/recall_at_5_macroAvg": "valfn_daniel_dedup/recall_at_5_macroAvg"}, inplace=True)

df["run_config"] = df.index.map(lambda x: x.rsplit("_", 1)[0])
# df["seed"] = df.index.map(lambda x: x.rsplit("_", 1)[1]).astype(int)
df = df.melt(var_name="metric", value_name="value", id_vars=["run_config"])

# 0-1 normalize each metric across all configs and seeds
df["normalized_value"] = df.groupby("metric")["value"].transform(lambda x: (x - x.min()) / (x.max() - x.min()))


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate means
means = df.groupby('run_config')['normalized_value'].mean().reset_index()

# Sort the means
sorted_means = means.sort_values('normalized_value')

# Order the DataFrame based on the sorted means
df['run_config'] = pd.Categorical(df['run_config'], categories=sorted_means['run_config'], ordered=True)
df = df.sort_values('run_config')

# Now plot with seaborn
fig, ax = plt.subplots(figsize=(4, 4))
sns.violinplot(data=df, x="run_config", y="normalized_value", ax=ax, color='gray')

# Set the x-tick labels with rotation
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

# Use tight_layout to adjust the plot
plt.tight_layout()

fig.savefig(snakemake.output.all_models_comparison)

In [ ]:
import matplotlib.pyplot as plt
import seaborn

# Assuming df and get_path are defined elsewhere in your code
# matplotlib.style.use(get_path(["plot_style"]))
fig, ax = plt.subplots(figsize=(4, 4))

df["run_config"] = df["run_config"].astype(str)

seaborn.lineplot(
    data=df[df.run_config.isin(["full_model", "no_archs4_data", "no_census_data", "no_sample_weights"])],
    x="metric",
    y="normalized_value",
    hue="run_config",
    ax=ax
)

ax.set(xlabel="", ylabel="Normalized value", ylim=(0, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

# Set the legend to display in multiple columns, for example, 2 columns
legend = ax.legend(ncol=2, loc='lower center', fancybox=True)


plt.tight_layout()
fig.savefig(snakemake.output.top_models_metrics_details)
